# Эксперимент 1: автоматические признаки DFS

Проверяю, может ли DFS заменить ручные признаки с потерей NDCG@10 менее 3%.
Сравню наборы, посчитаю bootstrap, проверю пять временных окон и Optuna.
Здесь ещё используется восстановленный `start_ts` с найденным каналом утечки.
Основной результат — в [итоговом сравнении](final-comparison.ipynb).

`report` показывает исходные результаты, `recompute` повторяет вычисления
под новым RUN_NAME.


Модель, целевые строки, группы и метрики одинаковые; меняется набор входов.


In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'task_1/src').is_dir() and (path / 'docs/results').is_dir())
TASK = ROOT / 'task_1'
sys.path.insert(0, str(TASK / 'src'))
from notebook_workflow import from_environment

MODE = os.getenv('TLAB_NOTEBOOK_MODE', 'report')  # report или recompute
RUN_NAME = os.getenv('TLAB_NOTEBOOK_RUN', 'repeat_01')
DOWNLOAD_DATA = os.getenv('TLAB_NOTEBOOK_DOWNLOAD', '1') == '1'
INCLUDE_TEST = os.getenv('TLAB_INCLUDE_TEST', '0') == '1'
workflow = from_environment(MODE, RUN_NAME, DOWNLOAD_DATA, INCLUDE_TEST)

workflow.ensure('dfs130')
DFS = workflow.path('dfs130')
ROLLING = workflow.path('rolling')
TUNING = workflow.path('optuna')
config = json.loads((DFS / 'run_config.json').read_text())
assert config['test_used'] is False
print('Режим:', MODE, 'результаты:', DFS.relative_to(ROOT))


Режим: report результаты: task_1/results/experiment-1/dfs


## Сущности и история

Для признаков нужна прошлая история, в том числе до train и в GAP.
Текущий label в модель не подаётся. ID нужны для связей между таблицами,
как числовые признаки в CatBoost их не использую.

In [2]:
entities = pd.DataFrame([
    {'Сущность': 'Users', 'Ключ': 'uid', 'История': 'Все взаимодействия пользователя'},
    {'Сущность': 'Items', 'Ключ': 'item_id', 'История': 'Все взаимодействия с треком'},
    {'Сущность': 'UserItem', 'Ключ': '(uid, item_id)', 'История': 'Повторные взаимодействия пары'},
    {'Сущность': 'UserArtist', 'Ключ': '(uid, artist_id)', 'История': 'Взаимодействия с исполнителем'},
])
display(entities)

,Сущность,Ключ,История
0,Users,uid,Все взаимодействия пользователя
1,Items,item_id,Все взаимодействия с треком
2,UserItem,"(uid, item_id)",Повторные взаимодействия пары
3,UserArtist,"(uid, artist_id)",Взаимодействия с исполнителем


`log_id` различает строки, включая одинаковые события.
Для истории использовалось `end_ts < start_ts`.
Проблема самого start_ts разобрана в [аудите времени](../../docs/start-time-audit.md).


## Какие формулы составил DFS

Метод: [Kanter и Veeramachaneni, 2015](https://www.maxkanter.com/papers/DSAA_DSM_2015.pdf). Реализация синтеза: [Featuretools](https://docs.featuretools.com/en/stable/generated/featuretools.dfs.html).

In [3]:
examples = []
for entity in entities['Сущность']:
    path = TASK / f'data/experiment-1/featuretools/train_sample_200_cumulative_{entity}_definitions.json'
    definitions = json.loads(path.read_text())
    for formula in definitions['feature_list'][:4]:
        examples.append({'Сущность': entity, 'Формула': formula})
display(pd.DataFrame(examples))
print('Модельных входов:', len(config['features']))

,Сущность,Формула
0,Users,Users: COUNT(Interactions)
1,Users,Users: MAX(Interactions.is_organic)
2,Users,Users: MAX(Interactions.label)
3,Users,Users: MAX(Interactions.track_length_seconds)
4,Items,Items: COUNT(Interactions)
5,Items,Items: MAX(Interactions.is_organic)
6,Items,Items: MAX(Interactions.label)
7,Items,Items: MAX(Interactions.track_length_seconds)
8,UserItem,UserItem: COUNT(Interactions)
9,UserItem,UserItem: MAX(Interactions.is_organic)


Модельных входов: 130


Глубину ограничил двумя уровнями. Использовал count, mean, std, min,
max и multiply_numeric. Из 103 формул каждой ветви взял первые 32
в порядке Featuretools, без отбора по качеству. Получилось 128 формул
и два исходных поля, всего 130 входов.

Featuretools составляет формулы. Значения считает накопительный движок,
чтобы переиспользовать историю между запросами. Он поддерживает выбранные
операции, а не все возможности библиотеки.

Сравнение с обычным Featuretools проведено на 200 реальных запросах.
Файл на 5000 строк — отдельный регрессионный снимок старого исполнителя.

## Исходное сравнение на полной validation

In [4]:
comparison = pd.read_csv(DFS / 'validation_comparison.csv', float_precision='round_trip')
display(comparison.sort_values('NDCG@10', ascending=False).reset_index(drop=True).round(6))

,model,NDCG@10,QueryAUC,logloss
0,catboost YetiRank,0.768618,0.611703,NaN
1,DFS YetiRank,0.760151,0.607175,NaN
2,DFS MultiClass E[label],0.753892,0.600354,0.557831
3,"catboost MultiClass, E[label]",0.748124,0.593861,0.585730
4,user track decay 7d,0.720290,0.548940,NaN
5,user listened track 30d,0.716998,0.540689,NaN
6,user listened artist 30d,0.708656,0.543069,NaN
7,user listened track recently,0.694645,0.542804,NaN
8,user liked track,0.691493,0.514042,NaN
9,item popularity 7d,0.688738,0.522527,NaN


При лимите 600 деревьев и early stopping 50 ручной YetiRank получил
NDCG@10 0.76862, DFS — 0.76015. Для MultiClass получилось 0.74812 и 0.75389.
У классификатора score равен P(1) + 2P(2), а log loss использует все
три вероятности. Группы состоят из наблюдавшихся событий user-day.

Здесь сравниваются два набора целиком. Отдельный эффект автоматизации
от остальных различий не выделен. После проверки времени эти числа
относятся только к старому cutoff.

## Время и сохранённые важности

In [5]:
times = []
for objective, record in config['models'].items():
    times.append({'Модель': objective, 'Деревьев': record['tree_count'],
                  'Потоков': record['training_threads'],
                  'Fit, мин': record['fit_seconds'] / 60})
display(pd.DataFrame(times).round(3))
importance = pd.read_csv(DFS / 'ranker_feature_importance.csv')
display(importance.head(15).round(5))

,Модель,Деревьев,Потоков,"Fit, мин"
0,ranker,532,4,40.189
1,pointwise,600,8,19.877


,Feature Id,Importances
0,UserItem__MEAN(Interactions.label),20.25078
1,Users__MEAN(Interactions.label),9.69405
2,UserItem__STD(Interactions.label),8.09718
3,UserArtist__MEAN(Interactions.label),5.03426
4,Items__MEAN(Interactions.label),4.74023
5,UserItem__COUNT(Interactions) * MEAN(Interacti...,3.73106
6,is_organic,3.71342
7,UserItem__COUNT(Interactions),2.51889
8,UserItem__STD(Interactions.is_organic * label),2.02237
9,track_length_seconds,2.00950


В таблице сохранённые важности исходного ранкера.
Состав нового набора разобран в итоговом ноутбуке.


## Допуск потери 3%

In [6]:
workflow.ensure('bootstrap')
bootstrap = pd.read_csv(DFS / 'bootstrap_noninferiority/summary.csv', float_precision='round_trip')
display(bootstrap[['objective', 'block_days', 'base_ndcg', 'dfs_ndcg',
                   'relative_loss', 'upper_bound', 'reject_h0']].round(6))


,objective,block_days,base_ndcg,dfs_ndcg,relative_loss,upper_bound,reject_h0
0,ranker,3,0.768618,0.760151,0.011016,0.011761,True
1,ranker,4,0.768618,0.760151,0.011016,0.011677,True
2,pointwise,3,0.748124,0.753892,-0.007709,-0.006432,True
3,pointwise,4,0.748124,0.753892,-0.007709,-0.006569,True


Относительная потеря равна `(NDCG_manual − NDCG_DFS) / NDCG_manual`.
H0: потеря не меньше 3%. Использую парный циклический bootstrap по дням:
блоки 3/4 дня, 20 000 повторов, seed 42. H0 отвергается, когда односторонняя
верхняя 95% граница ниже 0.03. В исходном запуске условие выполнено.


## Пять дополнительных временных окон

In [7]:
workflow.ensure('rolling')
rolling = pd.read_csv(ROLLING / 'summary.csv', float_precision='round_trip')
display(rolling[['window', 'objective', 'block_days', 'base_ndcg', 'dfs_ndcg',
                 'upper_bound', 'reject_h0']].round(6))


,window,objective,block_days,base_ndcg,dfs_ndcg,upper_bound,reject_h0
0,train_121_165,ranker,3,0.772357,0.765449,0.009816,True
1,train_121_165,ranker,4,0.772357,0.765449,0.009637,True
2,train_121_165,pointwise,3,0.752595,0.759803,-0.008141,True
3,train_121_165,pointwise,4,0.752595,0.759803,-0.008274,True
4,train_136_180,ranker,3,0.770826,0.762766,0.011204,True
5,train_136_180,ranker,4,0.770826,0.762766,0.011208,True
6,train_136_180,pointwise,3,0.751150,0.758261,-0.007881,True
7,train_136_180,pointwise,4,0.751150,0.758261,-0.007940,True
8,train_151_195,ranker,3,0.770442,0.762883,0.010651,True
9,train_151_195,ranker,4,0.770442,0.762883,0.010497,True


В каждом окне 45 дней train и следующие 15 дней validation.
Использованы исходные настройки 600/50. Во всех пяти исходных окнах
верхние границы потери ниже 3%. Окна пересекаются.


## Что дал подбор CatBoost

In [8]:
workflow.ensure('optuna')
tuned = pd.read_csv(TUNING / 'summary.csv', float_precision='round_trip')
columns = ['configuration', 'objective', 'manual_NDCG@10', 'dfs_NDCG@10',
           'ndcg_difference', 'relative_gain']
display(tuned[columns].round(6))


,configuration,objective,manual_NDCG@10,dfs_NDCG@10,ndcg_difference,relative_gain
0,fixed_600,ranker,0.768618,0.760151,-0.008467,-0.011016
1,fixed_600,pointwise,0.748124,0.753892,0.005767,0.007709
2,control_1600,ranker,0.769794,0.760151,-0.009642,-0.012526
3,control_1600,pointwise,0.749934,0.755510,0.005576,0.007435
4,tuned_1600,ranker,0.769328,0.761209,-0.008119,-0.010554
5,tuned_1600,pointwise,0.749934,0.755510,0.005576,0.007435


Сначала увеличил лимиты до 1600/80, затем Optuna подбирала depth,
learning rate и L2. В исходных 40 пробах лучшим ручным ранкером остался
контроль с NDCG@10 0.769794; DFS после подбора получил 0.761209.
[Параметры и результаты](../../docs/optuna.md).


## Сколько занимает составление формул

In [9]:
if MODE == 'recompute':
    workflow.ensure('formula-benchmark')
    benchmark_path = DFS / 'formula_construction_benchmark.json'
else:
    benchmark_path = DFS / 'formula_construction_benchmark_20261022T082802748474Z.json'
benchmark = json.loads(benchmark_path.read_text())
display(pd.Series({
    'Число формул': benchmark['generated_formula_count'],
    'Первое составление, с': benchmark['first_total_seconds'],
    'Медиана повторов, с': benchmark['median_total_seconds'],
}, name='Построение формул').to_frame())


,Построение формул
Число формул,128.000000
"Первое составление, с",0.021195
"Медиана повторов, с",0.025126


Замер относится к составлению 128 формул, без расчёта их значений.
Продолжение с окнами 1–30 дней — [эксперимент 2](experiment-2.ipynb).
